# Reproduce the player churn project

Run this notebook from the submission folder first. It uses the public implementation at the commit identified in the report. The checkout and raw data stay in a working directory. Submit the notebooks and report files only. Git and Python 3.11 were used for verification. See the README for dependencies and file sizes.

In [1]:
import hashlib
import os
from pathlib import Path
import subprocess

REPOSITORY = "https://github.com/BhanuTeja-24/player-churn-prediction.git"
COMMIT = "04cc2e674651741689973cb7027cd62abc6165ca"
ROOT = Path(os.environ.get("CHURN_PROJECT_ROOT", ".pipeline")).resolve()
if not (ROOT / ".git").is_dir():
    subprocess.run(["git", "clone", REPOSITORY, str(ROOT)], check=True)
    subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", COMMIT], check=True)
actual = subprocess.check_output(
    ["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True
).strip()
if actual != COMMIT:
    raise RuntimeError(f"Expected pipeline commit {COMMIT}. Found {actual}.")
print("Pinned pipeline checkout verified.")

Pinned pipeline checkout verified.


Choose the datasets to fetch. The three original analyses need all games. The new chronological notebook can be run with Game 1 alone. The archives are downloaded to local disk and verified against Zenodo checksums.

In [2]:
import shutil
import json
from urllib.request import urlopen

GAMES = ("game1", "game2", "game3")
ARCHIVES = {
    "game1": "rawdata_game1.csv",
    "game2": "rawdata_game2.zip",
    "game3": "rawdata_game3.zip",
}
with urlopen("https://zenodo.org/api/records/166035", timeout=30) as response:
    record = json.load(response)
indexed = {entry["key"]: entry for entry in record["files"]}
raw = ROOT / "data" / "raw"
raw.mkdir(parents=True, exist_ok=True)

def md5(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "md5").hexdigest()

for game in GAMES:
    entry = indexed[ARCHIVES[game]]
    expected = entry["checksum"].split(":", 1)[1]
    target = raw / entry["key"]
    if not target.is_file() or target.stat().st_size != entry["size"] or md5(target) != expected:
        pending = target.with_suffix(target.suffix + ".part")
        with urlopen(entry["links"]["self"], timeout=120) as response, pending.open("wb") as stream:
            shutil.copyfileobj(response, stream)
        if pending.stat().st_size != entry["size"] or md5(pending) != expected:
            raise IOError(f"Checksum or size mismatch for {entry['key']}")
        pending.replace(target)
    print(f"Verified {target.name}")

Verified rawdata_game1.csv


Verified rawdata_game2.zip


Verified rawdata_game3.zip


After installing the dependencies, use Notebook 01 to explore the raw data and prepare all games. To run only the Game 1 follow-up, set GAMES to ('game1',) above and PREPARE_SELECTED_GAMES to True below. Notebook 07 reads the outputs in the pinned repository. Set REBUILD_CROSS_GAME_OUTPUTS to True to regenerate them with all three games. That run is optional and can take considerable time.

In [3]:
import sys

PREPARE_SELECTED_GAMES = False
REBUILD_CROSS_GAME_OUTPUTS = False

if REBUILD_CROSS_GAME_OUTPUTS and set(GAMES) != set(ARCHIVES):
    raise ValueError("A cross-game rebuild requires all three archives.")

if PREPARE_SELECTED_GAMES or REBUILD_CROSS_GAME_OUTPUTS:
    for game in GAMES:
        if not (ROOT / "data" / "interim" / f"{game}_events.parquet").is_file():
            subprocess.run([sys.executable, "-m", "src.etl", "--game", game], cwd=ROOT, check=True)

if REBUILD_CROSS_GAME_OUTPUTS:
    subprocess.run([sys.executable, "run_pipeline.py"], cwd=ROOT, check=True)
else:
    print("Notebook 07 will use available cross-game artefacts from the pipeline checkout.")


Notebook 07 will use available cross-game artefacts from the pipeline checkout.
